In [0]:

dim_value = dbutils.jobs.taskVaules.get(
    taskKey ="dim_transformation",
    key = "dim_value",
    defaultValue = 0
)

fact_value = dbutils.jobs.taskVaules.get(
    taskKey ="fact_transformation",
    key = "fact_value",
    defaultValue = 0
)

total_value = int(dim_value) + int(fact_value)
print(f"Total value is {total_value}")

In [0]:
dbutils.widgets.text('user_name','Admin')
user_name = dbutils.widgets.get('user_name')
print(user_name)

In [0]:
status_df = spark.sql(f"""
SELECT
  'dim_value' as name,
  '{dim_value}' as value
UNION ALL
SELECT
  'fact_value' as name,
  '{fact_value}' as value
""")
display(status_df)



In [0]:
display(user_name)

In [0]:
%sql
use catalog tpcds_lakehouse;

In [0]:
%sql
-- Step 1: Run once to create the table (first load)
CREATE TABLE IF NOT EXISTS gold.dim_date
USING DELTA
CLUSTER BY (year, month)
AS
SELECT
    d_date_sk                                               AS date_sk,
    d_date                                                  AS full_date,
    d_year                                                  AS year,
    d_qoy                                                   AS quarter,
    d_moy                                                   AS month,
    d_dom                                                   AS day_of_month,
    d_dow                                                   AS day_of_week,
    d_day_name                                              AS day_name,
    d_week_seq                                              AS week_seq,
    d_month_seq                                             AS month_seq,
    d_quarter_seq                                           AS quarter_seq,
    CASE WHEN d_holiday = 'Y' THEN true ELSE false END      AS is_holiday,
    CASE WHEN d_weekend = 'Y' THEN true ELSE false END      AS is_weekend,
    CASE WHEN d_current_year = 'Y' THEN true ELSE false END AS is_current_year
FROM silver.date_dim
WHERE 1=0; -- empty shell, schema only

-- Step 2: Run every pipeline execution (incremental upsert)
MERGE INTO gold.dim_date AS target
USING (
    SELECT
        d_date_sk                                               AS date_sk,
        d_date                                                  AS full_date,
        d_year                                                  AS year,
        d_qoy                                                   AS quarter,
        d_moy                                                   AS month,
        d_dom                                                   AS day_of_month,
        d_dow                                                   AS day_of_week,
        d_day_name                                              AS day_name,
        d_week_seq                                              AS week_seq,
        d_month_seq                                             AS month_seq,
        d_quarter_seq                                           AS quarter_seq,
        CASE WHEN d_holiday = 'Y' THEN true ELSE false END      AS is_holiday,
        CASE WHEN d_weekend = 'Y' THEN true ELSE false END      AS is_weekend,
        CASE WHEN d_current_year = 'Y' THEN true ELSE false END AS is_current_year
    FROM silver.date_dim
) AS source
ON target.date_sk = source.date_sk

-- Update if any attribute changed (e.g. holiday flag corrected)
WHEN MATCHED AND (
    target.is_holiday   <> source.is_holiday  OR
    target.is_weekend   <> source.is_weekend  OR
    target.is_current_year <> source.is_current_year
) THEN UPDATE SET *

-- Insert net-new dates only
WHEN NOT MATCHED THEN INSERT *;